In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib as mpl
import geopandas as gpd
import sqlite3 as sq
import sys

%load_ext autoreload
%autoreload 2

from src.functions import read_census_table, categorize

In [2]:
ACS52024_1 = read_census_table("ACS5_2024_ZCTA_1")
ACS52024_2 = read_census_table("ACS5_2024_ZCTA_2")

In [3]:
data = ACS52024_1.merge(ACS52024_2, on = ['NAME', 'GEO_ID', 'ZCTA', 'GEO_ID_2'])

In [4]:
data['ZCTA'] = data['GEO_ID'].str[-5:]

In [6]:
cols = ['NAME', 'ZCTA', 'Population ACS',         
        #poverty
        'Poverty: Population Below Poverty Level', 
        #housing cost burden
        'CB:Cost Burdened Households',
        #unemployed
        'Employment:Unemployed Civilian'
       ]
df = data[cols]
#EC (economic)
df = df.rename(columns = {'Population ACS': 'POP',
                          'Poverty: Population Below Poverty Level': 'EC_POV', 
                          'CB:Cost Burdened Households': 'EC_CB', 
                          'Employment:Unemployed Civilian': 'EC_UNEMP'})

In [7]:
#use population to weight
weights_column = 'POP'

#function: calc weighted mean and st dev
def weighted_stats(column, weights):
    #ensure cols and weights numeric, replacing non-numeric values with nan
    column = pd.to_numeric(column, errors='coerce')
    weights = pd.to_numeric(weights, errors='coerce')
    
    #drop nan values in column or weights
    valid = ~column.isna() & ~weights.isna()
    column = column[valid]
    weights = weights[valid]
    
    #if weights sum to zero, raise error
    if weights.sum() == 0:
        raise ZeroDivisionError("Weights sum to zero, errorrrr.")
    
    #calc weighted mean and st dev
    weighted_mean = np.average(column, weights=weights)
    variance = np.average((column - weighted_mean)**2, weights=weights)
    weighted_std = np.sqrt(variance)
    return weighted_mean, weighted_std

In [8]:
to_classify = ['EC_POV', 'EC_CB', 'EC_UNEMP']

#iterate over cols and create new ranking columns
for col in to_classify:
    try:
        mean, std = weighted_stats(df[col], df[weights_column])
        rank_col = f'{col}_R' #R (rank)
        df[rank_col] = pd.to_numeric(df[col], errors='coerce').apply(
            lambda x: categorize(x, mean, std) if not pd.isna(x) else np.nan)
    except ZeroDivisionError:
        print(f"Skipping column {col} due to zero-sum weights or all NaN values.")
        continue

In [9]:
df.head()

,NAME,ZCTA,POP,EC_POV,EC_CB,EC_UNEMP,EC_POV_R,EC_CB_R,EC_UNEMP_R
0,ZCTA5 00601,00601,16721.0,10199.0,854.0,1301.0,3,1,2
1,ZCTA5 00602,00602,37510.0,17504.0,1700.0,756.0,4,2,2
2,ZCTA5 00603,00603,48317.0,22683.0,3636.0,2729.0,4,2,4
3,ZCTA5 00606,00606,5435.0,2984.0,233.0,0.0,2,1,1
4,ZCTA5 00610,00610,25413.0,11145.0,1408.0,625.0,3,2,2


In [10]:
#fill any nulls with 2; neutral
cols = ['EC_POV_R', 'EC_CB_R', 'EC_UNEMP_R']
df[cols] = df[cols].fillna(2)

In [12]:
#sum of economic ranks
columns_to_add = ['EC_POV_R', 'EC_CB_R', 'EC_UNEMP_R']
df['SUM_EC'] = df[columns_to_add].sum(axis=1)

In [13]:
#calculate the threshold for the top 40% of the sum of economic ranks
threshold = df['SUM_EC'].quantile(0.6)  #bottom 60% cutoff

#use this to create the 'EC_NEED' column
df['EC_NEED'] = (df['SUM_EC'] > threshold).astype(int)

In [14]:
df.head()

,NAME,ZCTA,POP,EC_POV,EC_CB,EC_UNEMP,EC_POV_R,EC_CB_R,EC_UNEMP_R,SUM_EC,EC_NEED
0,ZCTA5 00601,00601,16721.0,10199.0,854.0,1301.0,3,1,2,6,1
1,ZCTA5 00602,00602,37510.0,17504.0,1700.0,756.0,4,2,2,8,1
2,ZCTA5 00603,00603,48317.0,22683.0,3636.0,2729.0,4,2,4,10,1
3,ZCTA5 00606,00606,5435.0,2984.0,233.0,0.0,2,1,1,4,0
4,ZCTA5 00610,00610,25413.0,11145.0,1408.0,625.0,3,2,2,7,1


In [15]:
df['EC_NEED'].value_counts()

EC_NEED
0    20717
1    13055
Name: count, dtype: int64

In [16]:
#set economic need aside
economicneed = df

In [17]:
economicneed.head(2)

,NAME,ZCTA,POP,EC_POV,EC_CB,EC_UNEMP,EC_POV_R,EC_CB_R,EC_UNEMP_R,SUM_EC,EC_NEED
0,ZCTA5 00601,00601,16721.0,10199.0,854.0,1301.0,3,1,2,6,1
1,ZCTA5 00602,00602,37510.0,17504.0,1700.0,756.0,4,2,2,8,1


Social Need

In [18]:
#census bureau data points
cols = ['NAME', 'ZCTA', 'Population ACS', 
        'Age:65 and Older', #age, seniors
        'Minority ACS', #minority status
        'Disability:With Disability', #disability
        'Language:Language Besides English:Speak English Not Well or At All', #language
        'Vehicles:None' #no vehicles at household
       ]
data = data[cols]
data = data.fillna(0)
data['ZCTA'] = data['ZCTA'].astype(str)

In [19]:
#external USDA and EPA data from MySidewalk download
adj = pd.read_csv('../data/MySidewalk_SocialNeed.csv')
adj['ZCTA'] = adj['ZCTA'].astype(str)
adj.head(2)

,NAME,ZCTA,Environmental Hazard Expected Annual Loss,Diesel Particulate Matter Level in Air,Drinking Water Non-Compliance,Federal Office of Rural Health Policy - Rural Area,People 1 Miles Urban/10 Miles Rural with Low Access to Healthy Food
0,ZCTA5 37010,37010,567175.7283,0.1079,1.0,1,2380.0
1,ZCTA5 37012,37012,312750.1809,0.0670,6.0,3,6.0


In [20]:
#use the MySidewalk ZCTAs (selected in internet app manually) to filter down the Census ZCTAs. As they aren't nested in Census geos there's no simple filter and this will do.
zctas = list(adj['ZCTA'].unique())

In [21]:
#filter economic need and working social data
economicneed['ZCTA'] = economicneed['ZCTA'].astype(str)
economicneed = economicneed.loc[economicneed['ZCTA'].isin(zctas)]
data = data.loc[data['ZCTA'].isin(zctas)]

In [22]:
social = data.merge(adj, on = ['NAME', 'ZCTA'], how = 'left')

In [23]:
social.head(2)

,NAME,ZCTA,Population ACS,Age:65 and Older,Minority ACS,Disability:With Disability,Language:Language Besides English:Speak English Not Well or At All,Vehicles:None,Environmental Hazard Expected Annual Loss,Diesel Particulate Matter Level in Air,Drinking Water Non-Compliance,Federal Office of Rural Health Policy - Rural Area,People 1 Miles Urban/10 Miles Rural with Low Access to Healthy Food
0,ZCTA5 37010,37010,5143.0,879.0,982.0,928.0,7.0,66.0,567175.7283,0.1079,1.0,1,2380.0
1,ZCTA5 37012,37012,2149.0,331.0,168.0,396.0,0.0,37.0,312750.1809,0.0670,6.0,3,6.0


In [24]:
df = social.rename(columns = {'Population ACS': 'POP',
                            'Age:65 and Older': 'SC_AGE65', 
                            'Minority ACS': 'SC_MIN',
                            'Disability:With Disability': 'SC_DISABL', 
                            'Language:Language Besides English:Speak English Not Well or At All': 'SC_LIMENG', 
                            'Vehicles:None': 'SC_NOVEH', 
                            'Environmental Hazard Expected Annual Loss': 'SC_ENVHAZL', 
                            'Diesel Particulate Matter Level in Air': 'SC_DPML', 
                            'Drinking Water Non-Compliance': 'SC_WATER', 
                            'Federal Office of Rural Health Policy - Rural Area': 'SC_RURAL', 
                            'People 1 Miles Urban/10 Miles Rural with Low Access to Healthy Food': 'SC_FOOD'
                           })

In [25]:
df.head(2)

,NAME,ZCTA,POP,SC_AGE65,SC_MIN,SC_DISABL,SC_LIMENG,SC_NOVEH,SC_ENVHAZL,SC_DPML,SC_WATER,SC_RURAL,SC_FOOD
0,ZCTA5 37010,37010,5143.0,879.0,982.0,928.0,7.0,66.0,567175.7283,0.1079,1.0,1,2380.0
1,ZCTA5 37012,37012,2149.0,331.0,168.0,396.0,0.0,37.0,312750.1809,0.0670,6.0,3,6.0


In [26]:
df.columns

Index(['NAME', 'ZCTA', 'POP', 'SC_AGE65', 'SC_MIN', 'SC_DISABL', 'SC_LIMENG',
       'SC_NOVEH', 'SC_ENVHAZL', 'SC_DPML', 'SC_WATER', 'SC_RURAL', 'SC_FOOD'],
      dtype='str')

In [27]:
to_classify = ['SC_AGE65', 'SC_MIN', 'SC_DISABL', 'SC_LIMENG', 'SC_NOVEH', 'SC_ENVHAZL', 'SC_DPML', 'SC_WATER', 'SC_RURAL', 'SC_FOOD']
#already renamed, reset weights col
weights_column = 'POP'
#iterate over cols and create new ranking columns
for col in to_classify:
    try:
        mean, std = weighted_stats(df[col], df[weights_column])
        rank_col = f'{col}_R'
        df[rank_col] = pd.to_numeric(df[col], errors='coerce').apply(
            lambda x: categorize(x, mean, std) if not pd.isna(x) else np.nan
        )
    except ZeroDivisionError:
        print(f"Skipping column {col} due to zero-sum weights or all NaN values.")
        continue

In [28]:
cols = ['SC_AGE65_R', 'SC_MIN_R', 'SC_DISABL_R', 'SC_LIMENG_R', 'SC_NOVEH_R', 'SC_ENVHAZL_R', 'SC_DPML_R', 'SC_WATER_R', 'SC_RURAL_R', 'SC_FOOD_R']
#fillna with neutral
df[cols] = df[cols].fillna(2)

In [29]:
columns_to_add = ["SC_AGE65_R", "SC_MIN_R", "SC_DISABL_R", "SC_LIMENG_R", "SC_NOVEH_R", "SC_ENVHAZL_R", "SC_DPML_R", "SC_WATER_R", "SC_RURAL_R", "SC_FOOD_R"]

df["SUM_SC"] = df[columns_to_add].sum(axis=1)

In [30]:
#calculate the threshold for the top 40%
threshold = df['SUM_SC'].quantile(0.6)  #bottom 60% cutoff

#create the 'SC_NEED' column
df['SC_NEED'] = (df['SUM_SC'] > threshold).astype(int)

In [31]:
df.head(2)

,NAME,ZCTA,POP,SC_AGE65,SC_MIN,SC_DISABL,SC_LIMENG,SC_NOVEH,SC_ENVHAZL,SC_DPML,...,SC_DISABL_R,SC_LIMENG_R,SC_NOVEH_R,SC_ENVHAZL_R,SC_DPML_R,SC_WATER_R,SC_RURAL_R,SC_FOOD_R,SUM_SC,SC_NEED
0,ZCTA5 37010,37010,5143.0,879.0,982.0,928.0,7.0,66.0,567175.7283,0.1079,...,1,2,1,1,1.0,2.0,2,2.0,14.0,0
1,ZCTA5 37012,37012,2149.0,331.0,168.0,396.0,0.0,37.0,312750.1809,0.0670,...,1,2,1,1,1.0,3.0,2,1.0,14.0,0


In [32]:
df['SC_NEED'].value_counts()

SC_NEED
0    68
1    42
Name: count, dtype: int64

In [33]:
socialneed = df

In [34]:
combined = economicneed.merge(socialneed, on = ['ZCTA', 'NAME', 'POP'])

In [35]:
combined.head()

,NAME,ZCTA,POP,EC_POV,EC_CB,EC_UNEMP,EC_POV_R,EC_CB_R,EC_UNEMP_R,SUM_EC,...,SC_DISABL_R,SC_LIMENG_R,SC_NOVEH_R,SC_ENVHAZL_R,SC_DPML_R,SC_WATER_R,SC_RURAL_R,SC_FOOD_R,SUM_SC,SC_NEED
0,ZCTA5 37010,37010,5143.0,510.0,292.0,113.0,2,1,1,4,...,1,2,1,1,1.0,2.0,2,2.0,14.0,0
1,ZCTA5 37012,37012,2149.0,400.0,235.0,75.0,2,1,1,4,...,1,2,1,1,1.0,3.0,2,1.0,14.0,0
2,ZCTA5 37013,37013,102055.0,14499.0,12986.0,3931.0,4,4,4,12,...,3,4,3,4,2.0,2.0,2,3.0,29.0,1
3,ZCTA5 37014,37014,3633.0,186.0,304.0,0.0,2,1,1,4,...,1,2,1,1,2.0,2.0,2,1.0,14.0,0
4,ZCTA5 37015,37015,19547.0,1185.0,1595.0,379.0,2,2,2,6,...,2,2,1,1,1.0,2.0,4,1.0,18.0,1


In [36]:
combined['EC_SC_NEED'] = combined.apply(lambda row: 1 if row['EC_NEED'] + row['SC_NEED'] == 2 else 0, axis=1)

In [38]:
#write out results
combined.to_csv('../data/Economic_SocialNeed.csv', index = False)